In [ ]:
!pip install pandas -q

In [ ]:
import pandas as pd
from google.colab import files

SAMPLES_PER_CONDITION = 3
RANDOM_SEED = 42

# Keywords to match uploaded filenames — order matters (more specific first)
COMMUNITY_KEYWORDS = {
    "Dulong":     "dulong",
    "Miao-Hmong": "miao",
    "Lisu":       "lisu",
    "Wa":         "wa_",
    "Jingpo":     "jingpo",
    "Hani-Akha":  "hani",
    "Deang":      "deang",
    "Lahu":       "lahu",
}
# Dai-Thai excluded — IRR already completed

In [ ]:
print("Upload all raw response CSV files:")
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

In [ ]:
def find_file(keyword, uploaded_keys):
    """Find uploaded filename containing keyword (case-insensitive)"""
    for k in uploaded_keys:
        if keyword.lower() in k.lower():
            return k
    return None

def condition_label(row):
    m = "GPT" if "GPT" in str(row["model"]) else "DS"
    l = "ZH"  if row["language"] == "Chinese" else "EN"
    return f"{m}-{l}"

output_files = []

for community, keyword in COMMUNITY_KEYWORDS.items():
    filename = find_file(keyword, uploaded.keys())
    if filename is None:
        print(f"SKIPPED {community}: no file matching '{keyword}' found")
        continue

    df = pd.read_csv(filename)
    df["condition"] = df.apply(condition_label, axis=1)

    parts = []
    for cond in ["GPT-ZH", "GPT-EN", "DS-ZH", "DS-EN"]:
        subset = df[df["condition"] == cond]
        if len(subset) == 0:
            print(f"  WARNING: no rows for {cond} in {community}")
            continue
        parts.append(subset.sample(n=min(SAMPLES_PER_CONDITION, len(subset)), random_state=RANDOM_SEED))

    result = pd.concat(parts).reset_index(drop=True)
    result = result[["condition", "prompt_id", "model", "language", "prompt", "response"]]

    result["human_trans_border"]        = ""
    result["human_identity"]            = ""
    result["human_cultural_continuity"] = ""
    result["human_narrative"]           = ""
    result["human_notes"]               = ""

    safe = community.lower().replace("/","-").replace("'","").replace(" ","_")
    out  = f"annotation_{safe}.csv"
    result.to_csv(out, index=False, encoding="utf-8-sig")
    output_files.append(out)
    print(f"{community}: {len(result)} rows → {out}")

In [ ]:
for f in output_files:
    files.download(f)
    print(f"Downloaded: {f}")